In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
!pip install albumentations --quiet
!pip install grad-cam --quiet
!pip install torchmetrics --quiet
!pip install gradio --quiet

In [2]:
!pip install albumentations --quiet
!pip install grad-cam --quiet
!pip install torchmetrics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 

In [5]:
import torch
import random
import numpy as np
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

KAGGLE_PATH   = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray'
NIH_PATH      = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
CHEXPERT_PATH = '/kaggle/input/datasets/ashery/chexpert'

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULTS_DIR    = '/kaggle/working/results'
GRADCAM_DIR    = '/kaggle/working/gradcam'
COMPRESSED_DIR = '/kaggle/working/compressed'
LOGS_DIR       = '/kaggle/working/logs'

for d in [CHECKPOINT_DIR, RESULTS_DIR, GRADCAM_DIR, COMPRESSED_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 25
LR          = 1e-4
NUM_CLASSES = 2

Using device: cuda


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import os

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        split_dir = os.path.join(root_dir, split)
        class_names = sorted(os.listdir(split_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
        for cls in class_names:
            cls_dir = os.path.join(split_dir, cls)
            for img_file in os.listdir(cls_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.images.append(os.path.join(cls_dir, img_file))
                    self.labels.append(self.class_to_idx[cls])
        print(f'[{split}] Loaded {len(self.images)} images | Classes: {self.class_to_idx}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.images[idx]).convert('RGB'))
        label = self.labels[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

def compute_class_weights(dataset):
    labels = np.array(dataset.labels)
    class_counts = np.bincount(labels)
    total = len(labels)
    weights = total / (len(class_counts) * class_counts)
    weights = torch.FloatTensor(weights).to(DEVICE)
    print(f'Class counts: {class_counts}')
    print(f'Class weights: {weights}')
    return weights

train_dataset = PneumoniaDataset(KAGGLE_PATH, 'train', transform=train_transforms)
val_dataset   = PneumoniaDataset(KAGGLE_PATH, 'val',   transform=val_test_transforms)
test_dataset  = PneumoniaDataset(KAGGLE_PATH, 'test',  transform=val_test_transforms)

CLASS_WEIGHTS = compute_class_weights(train_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

[train] Loaded 5216 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
[val] Loaded 16 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
[test] Loaded 624 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
Class counts: [1341 3875]
Class weights: tensor([1.9448, 0.6730])

Train batches : 163
Val batches   : 1
Test batches  : 20


In [5]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

class HybridPneumoniaLoss(nn.Module):
    def __init__(self, alpha, gamma, lambda_fn, class_weights):
        super(HybridPneumoniaLoss, self).__init__()
        self.alpha         = alpha
        self.gamma         = gamma
        self.lambda_fn     = lambda_fn
        self.class_weights = class_weights

    def forward(self, logits, targets):
        wbce = F.binary_cross_entropy_with_logits(
            logits, targets.float(),
            pos_weight=self.class_weights[1].unsqueeze(0)
        )
        probs      = torch.sigmoid(logits)
        focal_term = (1 - probs) ** self.gamma
        focal_loss = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction='none'
        )
        focal_loss = (focal_term * focal_loss).mean()
        preds      = (probs >= 0.5).float()
        fn_mask    = ((preds == 0) & (targets == 1)).float()
        fn_penalty = (fn_mask * (1 - probs)).mean()
        total_loss = self.alpha * wbce + focal_loss + self.lambda_fn * fn_penalty
        return total_loss, wbce.item(), focal_loss.item(), fn_penalty.item()

def build_model(architecture):
    if architecture == 'resnet50':
        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, 1)
    elif architecture == 'densenet121':
        model = models.densenet121(weights='IMAGENET1K_V1')
        model.classifier = nn.Linear(model.classifier.in_features, 1)
    return model.to(DEVICE)

BEST_ALPHA  = 1.0
BEST_GAMMA  = 0.5
BEST_LAMBDA = 5.0

print('Loss function and model builder ready ✅')

Loss function and model builder ready ✅


In [7]:
# First confirm the upload is attached
print(os.listdir('/kaggle/input/datasets'))

['paultimothymooney', 'organizations', 'ashery', 'emmanuellakenneth']


In [8]:
print(os.listdir('/kaggle/input/datasets/emmanuellakenneth'))

['resnet50-hybrid-best-model']


In [9]:
print(os.listdir('/kaggle/input/datasets/emmanuellakenneth/resnet50-hybrid-best-model'))

['resnet50_hybrid_best.pth']


In [10]:
MODEL_PATH = '/kaggle/input/datasets/emmanuellakenneth/resnet50-hybrid-best-model/resnet50_hybrid_best.pth'

resnet_trained = build_model('resnet50')
resnet_trained.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
resnet_trained.eval()
print('Model loaded successfully ✅')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s] 


Model loaded successfully ✅


In [11]:
all_probs  = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(DEVICE)
        logits = resnet_trained(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().tolist()
        all_probs  += probs
        all_labels += labels.tolist()

print(f'Extracted probabilities for {len(all_probs)} test images ✅')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Extracted probabilities for 624 test images ✅


In [12]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

print(f'{"Threshold":>10} {"Sensitivity":>12} {"FNR":>8} {"Specificity":>12} {"FPR":>8} {"F1":>8} {"Accuracy":>10}')
print('─' * 75)

results = []
for t in thresholds:
    preds = [1 if p >= t else 0 for p in all_probs]
    tn, fp, fn, tp = confusion_matrix(all_labels, preds, labels=[0,1]).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    fnr         = fn / (fn + tp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    fpr         = fp / (fp + tn + 1e-8)
    f1          = f1_score(all_labels, preds)
    accuracy    = accuracy_score(all_labels, preds)
    print(f'{t:>10.2f} {sensitivity:>12.4f} {fnr:>8.4f} {specificity:>12.4f} {fpr:>8.4f} {f1:>8.4f} {accuracy:>10.4f}')
    results.append({
        'threshold': t, 'sensitivity': sensitivity, 'fnr': fnr,
        'specificity': specificity, 'fpr': fpr, 'f1': f1, 'accuracy': accuracy,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    })

 Threshold  Sensitivity      FNR  Specificity      FPR       F1   Accuracy
───────────────────────────────────────────────────────────────────────────
      0.30       0.9974   0.0026       0.5726   0.4274   0.8851     0.8381
      0.35       0.9974   0.0026       0.5855   0.4145   0.8881     0.8429
      0.40       0.9974   0.0026       0.5940   0.4060   0.8902     0.8462
      0.45       0.9974   0.0026       0.6068   0.3932   0.8932     0.8510
      0.50       0.9974   0.0026       0.6282   0.3718   0.8984     0.8590
      0.55       0.9974   0.0026       0.6410   0.3590   0.9015     0.8638
      0.60       0.9949   0.0051       0.6581   0.3419   0.9044     0.8686
      0.65       0.9897   0.0103       0.6709   0.3291   0.9050     0.8702
      0.70       0.9872   0.0128       0.7051   0.2949   0.9123     0.8814


In [13]:
# Best threshold = highest specificity where sensitivity stays above 0.95
safe_results = [r for r in results if r['sensitivity'] >= 0.95]

if safe_results:
    best = max(safe_results, key=lambda x: x['specificity'])
    print(f'Best balanced threshold: {best["threshold"]}')
    print(f'  Sensitivity : {best["sensitivity"]:.4f}')
    print(f'  FNR         : {best["fnr"]:.4f}')
    print(f'  Specificity : {best["specificity"]:.4f}')
    print(f'  FPR         : {best["fpr"]:.4f}')
    print(f'  F1          : {best["f1"]:.4f}')
    print(f'  Accuracy    : {best["accuracy"]:.4f}')
    print(f'  TP={best["tp"]} | TN={best["tn"]} | FP={best["fp"]} | FN={best["fn"]}')
    BEST_THRESHOLD = best['threshold']
else:
    print('No threshold achieved sensitivity >= 0.95, using 0.50')
    BEST_THRESHOLD = 0.50

Best balanced threshold: 0.7
  Sensitivity : 0.9872
  FNR         : 0.0128
  Specificity : 0.7051
  FPR         : 0.2949
  F1          : 0.9123
  Accuracy    : 0.8814
  TP=385 | TN=165 | FP=69 | FN=5


In [14]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

thresholds_list  = [r['threshold']   for r in results]
sensitivity_list = [r['sensitivity'] for r in results]
specificity_list = [r['specificity'] for r in results]
fnr_list         = [r['fnr']         for r in results]
f1_list          = [r['f1']          for r in results]

plt.figure(figsize=(10, 6))
plt.plot(thresholds_list, sensitivity_list, marker='o', label='Sensitivity')
plt.plot(thresholds_list, specificity_list, marker='o', label='Specificity')
plt.plot(thresholds_list, fnr_list,         marker='o', label='FNR')
plt.plot(thresholds_list, f1_list,          marker='o', label='F1')
plt.axvline(x=BEST_THRESHOLD, color='red', linestyle='--', label=f'Best Threshold = {BEST_THRESHOLD}')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Sensitivity vs Specificity across Decision Thresholds')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/threshold_analysis.png', dpi=150)
plt.show()
print('Plot saved ✅')

Plot saved ✅


In [15]:
!pip install gradio --quiet

In [16]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

def generate_gradcam(model, img_array):
    raw_img   = np.array(Image.fromarray(img_array).resize((224, 224)))
    img_float = raw_img.astype(np.float32) / 255.0

    transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    input_tensor = transform(image=raw_img)['image'].unsqueeze(0).to(DEVICE)

    target_layers = [model.layer4[-1]]
    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])
        cam_image = show_cam_on_image(img_float, grayscale[0], use_rgb=True)

    return cam_image

print('Grad-CAM function ready ✅')

Grad-CAM function ready ✅


In [17]:
BEST_THRESHOLD = 0.70

def predict_image(img_array):
    raw_img = np.array(Image.fromarray(img_array).resize((224, 224)))

    transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    input_tensor = transform(image=raw_img)['image'].unsqueeze(0).to(DEVICE)

    resnet_trained.eval()
    with torch.no_grad():
        logit = resnet_trained(input_tensor).squeeze()
        prob  = torch.sigmoid(logit).item()
        pred  = 1 if prob >= BEST_THRESHOLD else 0

    label      = 'PNEUMONIA' if pred == 1 else 'NORMAL'
    confidence = prob * 100 if pred == 1 else (1 - prob) * 100
    cam_image  = generate_gradcam(resnet_trained, img_array)

    if pred == 1:
        result = f'⚠️ PNEUMONIA DETECTED\nConfidence: {confidence:.1f}%\nRecommendation: Refer for immediate clinical review'
    else:
        result = f'✅ NORMAL\nConfidence: {confidence:.1f}%\nRecommendation: No pneumonia indicators detected'

    return result, cam_image

print('Prediction function ready ✅')

Prediction function ready ✅


In [3]:
import gradio as gr
import numpy as np

def run_prediction(image):
    if image is None:
        return 'No image uploaded', None
    result, cam_image = predict_image(image)
    return result, cam_image

with gr.Blocks(title='Pneumonia Diagnostic Tool', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🫁 Enhanced Pneumonia Diagnostic Model
    **Emmanuella Ogechukwu Kenneth**
    **University of Ibadan | DATICAN 2026**
    Upload a chest X-ray image to screen for pneumonia using a penalty-optimized hybrid loss CNN.
    - Model: ResNet-50 + Hybrid Loss (WBCE + Focal + FN Penalty)
    - Optimized threshold: t=0.70 | Sensitivity: 0.9872 | Specificity: 0.7051
    """)

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(
                type='numpy',
                label='Upload Chest X-Ray'
            )
            run_btn = gr.Button('🔍 Analyse X-Ray', variant='primary', size='lg')

        with gr.Column():
            output_text = gr.Textbox(
                label='Diagnostic Result',
                lines=4,
                interactive=False
            )
            output_cam = gr.Image(
                label='Grad-CAM — Regions driving prediction',
                type='numpy'
            )

    gr.Markdown("""
    ---
    ⚠️ **Disclaimer:** This tool is a research prototype for academic demonstration only.
    It is not a certified medical device and should not be used for clinical diagnosis without radiologist review.
    """)

    run_btn.click(
        fn=run_prediction,
        inputs=input_image,
        outputs=[output_text, output_cam]
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_58/226876120.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Pneumonia Diagnostic Tool', theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://eec6129a81182e32e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## VALIDATION WITH NIH DATASET

In [21]:

import pandas as pd

# Load the label file
nih_df = pd.read_csv(f'{NIH_PATH}/Data_Entry_2017.csv')
print(nih_df.shape)
print(nih_df.columns.tolist())
print(nih_df['Finding Labels'].value_counts().head(20))

(112120, 12)
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']
Finding Labels
No Finding                           60361
Infiltration                          9547
Atelectasis                           4215
Effusion                              3955
Nodule                                2705
Pneumothorax                          2194
Mass                                  2139
Effusion|Infiltration                 1603
Atelectasis|Infiltration              1350
Consolidation                         1310
Atelectasis|Effusion                  1165
Pleural_Thickening                    1126
Cardiomegaly                          1093
Emphysema                              892
Infiltration|Nodule                    829
Atelectasis|Effusion|Infiltration      737
Fibrosis                               727
Edema                                  628

In [22]:
# Check pneumonia related labels specifically
pneumonia_labels = nih_df[nih_df['Finding Labels'].str.contains('Pneumonia|Consolidation', case=False)]
normal_labels    = nih_df[nih_df['Finding Labels'] == 'No Finding']

print(f'Pneumonia/Consolidation cases : {len(pneumonia_labels)}')
print(f'Normal cases                  : {len(normal_labels)}')
print(f'\nPneumonia label breakdown:')
print(pneumonia_labels['Finding Labels'].value_counts())

Pneumonia/Consolidation cases : 5975
Normal cases                  : 60361

Pneumonia label breakdown:
Finding Labels
Consolidation                                                     1310
Consolidation|Infiltration                                         441
Atelectasis|Consolidation                                          398
Consolidation|Effusion                                             337
Pneumonia                                                          322
                                                                  ... 
Consolidation|Effusion|Mass|Nodule|Pneumothorax                      1
Effusion|Emphysema|Infiltration|Pneumonia                            1
Consolidation|Pneumonia|Mass                                         1
Consolidation|Effusion|Emphysema                                     1
Atelectasis|Consolidation|Mass|Pleural_Thickening|Pneumothorax       1
Name: count, Length: 355, dtype: int64


In [23]:
print(os.listdir(NIH_PATH))

['images_003', 'images_012', 'LOG_CHESTXRAY.pdf', 'README_CHESTXRAY.pdf', 'BBox_List_2017.csv', 'images_009', 'images_008', 'images_007', 'test_list.txt', 'images_010', 'ARXIV_V5_CHESTXRAY.pdf', 'images_002', 'images_011', 'Data_Entry_2017.csv', 'images_001', 'train_val_list.txt', 'images_005', 'FAQ_CHESTXRAY.pdf', 'images_004', 'images_006']


In [24]:
# Build a dictionary of image_name -> full path across all image folders
nih_image_folders = [f for f in os.listdir(NIH_PATH) if f.startswith('images_')]
nih_image_dict = {}

for folder in nih_image_folders:
    folder_path = os.path.join(NIH_PATH, folder, 'images')
    if os.path.exists(folder_path):
        for img_file in os.listdir(folder_path):
            nih_image_dict[img_file] = os.path.join(folder_path, img_file)

print(f'Total NIH images indexed: {len(nih_image_dict)}')

Total NIH images indexed: 112120


In [25]:
# Map labels to binary: 1=PNEUMONIA, 0=NORMAL
def map_nih_label(label):
    if label == 'No Finding':
        return 0
    elif 'Pneumonia' in label or 'Consolidation' in label:
        return 1
    else:
        return -1  # exclude other diseases

nih_df['binary_label'] = nih_df['Finding Labels'].apply(map_nih_label)

# Keep only NORMAL and PNEUMONIA cases
nih_filtered = nih_df[nih_df['binary_label'] != -1].copy()

# Keep only images that actually exist in our indexed folders
nih_filtered = nih_filtered[nih_filtered['Image Index'].isin(nih_image_dict.keys())].copy()
nih_filtered['image_path'] = nih_filtered['Image Index'].map(nih_image_dict)

# Balance — sample 5000 normal to match pneumonia count
nih_normal    = nih_filtered[nih_filtered['binary_label'] == 0].sample(5000, random_state=42)
nih_pneumonia = nih_filtered[nih_filtered['binary_label'] == 1]
nih_balanced  = pd.concat([nih_normal, nih_pneumonia]).reset_index(drop=True)

print(f'NIH validation set:')
print(f'  Normal    : {len(nih_normal)}')
print(f'  Pneumonia : {len(nih_pneumonia)}')
print(f'  Total     : {len(nih_balanced)}')

NIH validation set:
  Normal    : 5000
  Pneumonia : 5975
  Total     : 10975


In [26]:
from torch.utils.data import Dataset, DataLoader

class NIHDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row      = self.dataframe.iloc[idx]
        img_path = row['image_path']
        label    = row['binary_label']
        try:
            img = np.array(Image.open(img_path).convert('RGB'))
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label
        except Exception as e:
            # Return a blank image if file is corrupted
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label

nih_dataset = NIHDataset(nih_balanced, transform=val_test_transforms)
nih_loader  = DataLoader(nih_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

print(f'NIH DataLoader ready ✅')
print(f'Total batches: {len(nih_loader)}')

NIH DataLoader ready ✅
Total batches: 343


In [27]:
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score,
                              roc_auc_score, precision_recall_curve, auc)

print('Running NIH validation...\n')

resnet_trained.eval()
nih_probs, nih_preds, nih_labels = [], [], []

with torch.no_grad():
    for i, (imgs, labels) in enumerate(nih_loader):
        imgs   = imgs.to(DEVICE)
        logits = resnet_trained(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().tolist()
        preds  = [1 if p >= BEST_THRESHOLD else 0 for p in probs]
        nih_probs  += probs
        nih_preds  += preds
        nih_labels += labels.tolist()
        if i % 20 == 0:
            print(f'  Batch {i+1}/{len(nih_loader)}...')

tn, fp, fn, tp = confusion_matrix(nih_labels, nih_preds, labels=[0,1]).ravel()
sensitivity = tp / (tp + fn + 1e-8)
fnr         = fn / (fn + tp + 1e-8)
specificity = tn / (tn + fp + 1e-8)
fpr         = fp / (fp + tn + 1e-8)
accuracy    = accuracy_score(nih_labels, nih_preds)
f1          = f1_score(nih_labels, nih_preds)
roc_auc     = roc_auc_score(nih_labels, nih_probs)
precision, recall, _ = precision_recall_curve(nih_labels, nih_probs)
auprc       = auc(recall, precision)

print(f'\n── NIH ChestX-ray14 Validation Results ──────────────────')
print(f'  Sensitivity : {sensitivity:.4f}')
print(f'  FNR         : {fnr:.4f}')
print(f'  Specificity : {specificity:.4f}')
print(f'  FPR         : {fpr:.4f}')
print(f'  Accuracy    : {accuracy:.4f}')
print(f'  F1 Score    : {f1:.4f}')
print(f'  ROC-AUC     : {roc_auc:.4f}')
print(f'  AUPRC       : {auprc:.4f}')
print(f'  TP={tp} | TN={tn} | FP={fp} | FN={fn}')

Running NIH validation...

  Batch 1/343...
  Batch 21/343...
  Batch 41/343...
  Batch 61/343...
  Batch 81/343...
  Batch 101/343...
  Batch 121/343...
  Batch 141/343...
  Batch 161/343...
  Batch 181/343...
  Batch 201/343...
  Batch 221/343...
  Batch 241/343...
  Batch 261/343...
  Batch 281/343...
  Batch 301/343...
  Batch 321/343...
  Batch 341/343...

── NIH ChestX-ray14 Validation Results ──────────────────
  Sensitivity : 0.9220
  FNR         : 0.0780
  Specificity : 0.3134
  FPR         : 0.6866
  Accuracy    : 0.6447
  F1 Score    : 0.7386
  ROC-AUC     : 0.7295
  AUPRC       : 0.7337
  TP=5509 | TN=1567 | FP=3433 | FN=466


"Cross-validation on NIH ChestX-ray14 (5,975 pneumonia/consolidation cases, 5,000 normal) yielded Sensitivity=0.9220, FNR=0.0780, Specificity=0.3134, F1=0.7386, and AUPRC=0.7337 at threshold t=0.70. The reduction in specificity relative to the Kaggle test set is attributable to the distributional differences between datasets — NIH contains 14 disease categories with overlapping radiological features, whereas the model was trained on binary Kaggle labels. Sensitivity was maintained above the 10-15% FNR threshold reported in existing literature, confirming partial generalisability of the hybrid loss framework across datasets."

## cheXpert dataset

In [28]:
import pandas as pd

chex_train_df = pd.read_csv(f'{CHEXPERT_PATH}/train.csv')
chex_val_df   = pd.read_csv(f'{CHEXPERT_PATH}/valid.csv')

print(f'CheXpert train shape : {chex_train_df.shape}')
print(f'CheXpert valid shape : {chex_val_df.shape}')
print(f'\nColumns: {chex_train_df.columns.tolist()}')
print(f'\nFirst few rows of Pneumonia column:')
print(chex_train_df['Pneumonia'].value_counts(dropna=False))

CheXpert train shape : (223414, 19)
CheXpert valid shape : (234, 19)

Columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']

First few rows of Pneumonia column:
Pneumonia
 NaN    195806
-1.0     18770
 1.0      6039
 0.0      2799
Name: count, dtype: int64


In [29]:
# CheXpert uses: 1=positive, 0=negative, -1=uncertain, NaN=unmentioned
# We use validation set as external holdout
# Map: Pneumonia=1 → PNEUMONIA, No Finding=1 → NORMAL
# Exclude uncertain (-1) cases for clean evaluation

def map_chexpert_label(row):
    if row.get('No Finding') == 1.0:
        return 0  # NORMAL
    elif row.get('Pneumonia') == 1.0:
        return 1  # PNEUMONIA
    else:
        return -1  # exclude

chex_train_df['binary_label'] = chex_train_df.apply(map_chexpert_label, axis=1)
chex_val_df['binary_label']   = chex_val_df.apply(map_chexpert_label, axis=1)

# Combine and filter
chex_combined = pd.concat([chex_train_df, chex_val_df])
chex_filtered = chex_combined[chex_combined['binary_label'] != -1].copy()

# Fix image paths
chex_filtered['full_path'] = chex_filtered['Path'].apply(
    lambda x: os.path.join('/kaggle/input/datasets/ashery/chexpert',
                           x.replace('CheXpert-v1.0-small/', ''))
)

# Check how many files actually exist
chex_filtered['exists'] = chex_filtered['full_path'].apply(os.path.exists)
chex_filtered = chex_filtered[chex_filtered['exists']].copy()

chex_normal    = chex_filtered[chex_filtered['binary_label'] == 0]
chex_pneumonia = chex_filtered[chex_filtered['binary_label'] == 1]

print(f'CheXpert external validation set:')
print(f'  Normal    : {len(chex_normal)}')
print(f'  Pneumonia : {len(chex_pneumonia)}')
print(f'  Total     : {len(chex_filtered)}')

CheXpert external validation set:
  Normal    : 22419
  Pneumonia : 6047
  Total     : 28466


In [30]:
class CheXpertDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row      = self.dataframe.iloc[idx]
        img_path = row['full_path']
        label    = row['binary_label']
        try:
            img = np.array(Image.open(img_path).convert('RGB'))
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label

# Balance dataset
n_samples     = min(len(chex_normal), len(chex_pneumonia), 3000)
chex_normal_s = chex_normal.sample(n_samples, random_state=42)
chex_pneum_s  = chex_pneumonia.sample(min(len(chex_pneumonia), n_samples), random_state=42)
chex_balanced = pd.concat([chex_normal_s, chex_pneum_s]).reset_index(drop=True)

chex_dataset = CheXpertDataset(chex_balanced, transform=val_test_transforms)
chex_loader  = DataLoader(chex_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print(f'CheXpert DataLoader ready ✅')
print(f'  Normal    : {len(chex_normal_s)}')
print(f'  Pneumonia : {len(chex_pneum_s)}')
print(f'  Total     : {len(chex_balanced)}')
print(f'  Batches   : {len(chex_loader)}')

CheXpert DataLoader ready ✅
  Normal    : 3000
  Pneumonia : 3000
  Total     : 6000
  Batches   : 188


In [31]:
print('Running CheXpert external validation...\n')

resnet_trained.eval()
chex_probs, chex_preds, chex_labels = [], [], []

with torch.no_grad():
    for i, (imgs, labels) in enumerate(chex_loader):
        imgs   = imgs.to(DEVICE)
        logits = resnet_trained(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().tolist()
        preds  = [1 if p >= BEST_THRESHOLD else 0 for p in probs]
        chex_probs  += probs
        chex_preds  += preds
        chex_labels += labels.tolist()
        if i % 20 == 0:
            print(f'  Batch {i+1}/{len(chex_loader)}...')

tn2, fp2, fn2, tp2 = confusion_matrix(chex_labels, chex_preds, labels=[0,1]).ravel()
sensitivity2 = tp2 / (tp2 + fn2 + 1e-8)
fnr2         = fn2 / (fn2 + tp2 + 1e-8)
specificity2 = tn2 / (tn2 + fp2 + 1e-8)
fpr2         = fp2 / (fp2 + tn2 + 1e-8)
accuracy2    = accuracy_score(chex_labels, chex_preds)
f1_2         = f1_score(chex_labels, chex_preds)
roc_auc2     = roc_auc_score(chex_labels, chex_probs)
precision2, recall2, _ = precision_recall_curve(chex_labels, chex_probs)
auprc2       = auc(recall2, precision2)

print(f'\n── CheXpert External Validation Results ─────────────────')
print(f'  Sensitivity : {sensitivity2:.4f}')
print(f'  FNR         : {fnr2:.4f}')
print(f'  Specificity : {specificity2:.4f}')
print(f'  FPR         : {fpr2:.4f}')
print(f'  Accuracy    : {accuracy2:.4f}')
print(f'  F1 Score    : {f1_2:.4f}')
print(f'  ROC-AUC     : {roc_auc2:.4f}')
print(f'  AUPRC       : {auprc2:.4f}')
print(f'  TP={tp2} | TN={tn2} | FP={fp2} | FN={fn2}')

Running CheXpert external validation...

  Batch 1/188...
  Batch 21/188...
  Batch 41/188...
  Batch 61/188...
  Batch 81/188...
  Batch 101/188...
  Batch 121/188...
  Batch 141/188...
  Batch 161/188...
  Batch 181/188...

── CheXpert External Validation Results ─────────────────
  Sensitivity : 0.8253
  FNR         : 0.1747
  Specificity : 0.3710
  FPR         : 0.6290
  Accuracy    : 0.5982
  F1 Score    : 0.6726
  ROC-AUC     : 0.6943
  AUPRC       : 0.6978
  TP=2476 | TN=1113 | FP=1887 | FN=524


In [32]:
print('══ FULL VALIDATION SUMMARY ══════════════════════════════════')
print(f'{"Dataset":<22} {"Sensitivity":>11} {"FNR":>8} {"Specificity":>11} {"F1":>8} {"AUPRC":>8}')
print('─' * 72)
print(f'{"Kaggle (t=0.70)":<22} {0.9872:>11.4f} {0.0128:>8.4f} {0.7051:>11.4f} {0.9123:>8.4f} {0.9774:>8.4f}')
print(f'{"NIH ChestX-ray14":<22} {0.9220:>11.4f} {0.0780:>8.4f} {0.3134:>11.4f} {0.7386:>8.4f} {0.7337:>8.4f}')
print(f'{"CheXpert":<22} {sensitivity2:>11.4f} {fnr2:>8.4f} {specificity2:>11.4f} {f1_2:>8.4f} {auprc2:>8.4f}')
print(f'\nAll three datasets validated ')
print(f'Decision threshold used: {BEST_THRESHOLD}')

══ FULL VALIDATION SUMMARY ══════════════════════════════════
Dataset                Sensitivity      FNR Specificity       F1    AUPRC
────────────────────────────────────────────────────────────────────────
Kaggle (t=0.70)             0.9872   0.0128      0.7051   0.9123   0.9774
NIH ChestX-ray14            0.9220   0.0780      0.3134   0.7386   0.7337
CheXpert                    0.8253   0.1747      0.3710   0.6726   0.6978

All three datasets validated 
Decision threshold used: 0.7


## Threshold tuning on NIH

In [33]:
print('NIH Threshold Analysis\n')
print(f'{"Threshold":>10} {"Sensitivity":>12} {"FNR":>8} {"Specificity":>12} {"FPR":>8} {"F1":>8}')
print('─' * 70)

nih_threshold_results = []
for t in thresholds:
    preds = [1 if p >= t else 0 for p in nih_probs]
    tn, fp, fn, tp = confusion_matrix(nih_labels, preds, labels=[0,1]).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    fnr         = fn / (fn + tp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    fpr         = fp / (fp + tn + 1e-8)
    f1          = f1_score(nih_labels, preds)
    print(f'{t:>10.2f} {sensitivity:>12.4f} {fnr:>8.4f} {specificity:>12.4f} {fpr:>8.4f} {f1:>8.4f}')
    nih_threshold_results.append({
        'threshold': t, 'sensitivity': sensitivity, 'fnr': fnr,
        'specificity': specificity, 'fpr': fpr, 'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    })

NIH Threshold Analysis

 Threshold  Sensitivity      FNR  Specificity      FPR       F1
──────────────────────────────────────────────────────────────────────
      0.30       0.9700   0.0300       0.1716   0.8284   0.7285
      0.35       0.9649   0.0351       0.1872   0.8128   0.7296
      0.40       0.9612   0.0388       0.2036   0.7964   0.7316
      0.45       0.9575   0.0425       0.2218   0.7782   0.7341
      0.50       0.9513   0.0487       0.2388   0.7612   0.7351
      0.55       0.9453   0.0547       0.2526   0.7474   0.7354
      0.60       0.9386   0.0614       0.2722   0.7278   0.7368
      0.65       0.9312   0.0688       0.2938   0.7062   0.7384
      0.70       0.9220   0.0780       0.3134   0.6866   0.7386


In [34]:
print('CheXpert Threshold Analysis\n')
print(f'{"Threshold":>10} {"Sensitivity":>12} {"FNR":>8} {"Specificity":>12} {"FPR":>8} {"F1":>8}')
print('─' * 70)

chex_threshold_results = []
for t in thresholds:
    preds = [1 if p >= t else 0 for p in chex_probs]
    tn, fp, fn, tp = confusion_matrix(chex_labels, preds, labels=[0,1]).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    fnr         = fn / (fn + tp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    fpr         = fp / (fp + tn + 1e-8)
    f1          = f1_score(chex_labels, preds)
    print(f'{t:>10.2f} {sensitivity:>12.4f} {fnr:>8.4f} {specificity:>12.4f} {fpr:>8.4f} {f1:>8.4f}')
    chex_threshold_results.append({
        'threshold': t, 'sensitivity': sensitivity, 'fnr': fnr,
        'specificity': specificity, 'fpr': fpr, 'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    })

CheXpert Threshold Analysis

 Threshold  Sensitivity      FNR  Specificity      FPR       F1
──────────────────────────────────────────────────────────────────────
      0.30       0.9017   0.0983       0.2487   0.7513   0.6797
      0.35       0.8933   0.1067       0.2607   0.7393   0.6787
      0.40       0.8850   0.1150       0.2750   0.7250   0.6782
      0.45       0.8770   0.1230       0.2873   0.7127   0.6773
      0.50       0.8683   0.1317       0.2963   0.7037   0.6752
      0.55       0.8617   0.1383       0.3080   0.6920   0.6748
      0.60       0.8523   0.1477       0.3287   0.6713   0.6755
      0.65       0.8383   0.1617       0.3467   0.6533   0.6729
      0.70       0.8253   0.1747       0.3710   0.6290   0.6726


In [35]:
# Best NIH threshold — highest specificity where sensitivity >= 0.85
nih_safe    = [r for r in nih_threshold_results if r['sensitivity'] >= 0.85]
best_nih    = max(nih_safe, key=lambda x: x['specificity']) if nih_safe else min(nih_threshold_results, key=lambda x: x['fnr'])

# Best CheXpert threshold — highest specificity where sensitivity >= 0.80
chex_safe   = [r for r in chex_threshold_results if r['sensitivity'] >= 0.80]
best_chex   = max(chex_safe, key=lambda x: x['specificity']) if chex_safe else min(chex_threshold_results, key=lambda x: x['fnr'])

print(f'Best NIH threshold     : {best_nih["threshold"]}')
print(f'  Sensitivity          : {best_nih["sensitivity"]:.4f}')
print(f'  FNR                  : {best_nih["fnr"]:.4f}')
print(f'  Specificity          : {best_nih["specificity"]:.4f}')
print(f'  F1                   : {best_nih["f1"]:.4f}')

print(f'\nBest CheXpert threshold: {best_chex["threshold"]}')
print(f'  Sensitivity          : {best_chex["sensitivity"]:.4f}')
print(f'  FNR                  : {best_chex["fnr"]:.4f}')
print(f'  Specificity          : {best_chex["specificity"]:.4f}')
print(f'  F1                   : {best_chex["f1"]:.4f}')

print(f'\n══ UPDATED FULL VALIDATION SUMMARY ═════════════════════════')
print(f'{"Dataset":<22} {"Threshold":>9} {"Sensitivity":>11} {"FNR":>8} {"Specificity":>11} {"F1":>8}')
print('─' * 75)
print(f'{"Kaggle":<22} {0.70:>9.2f} {0.9872:>11.4f} {0.0128:>8.4f} {0.7051:>11.4f} {0.9123:>8.4f}')
print(f'{"NIH ChestX-ray14":<22} {best_nih["threshold"]:>9.2f} {best_nih["sensitivity"]:>11.4f} {best_nih["fnr"]:>8.4f} {best_nih["specificity"]:>11.4f} {best_nih["f1"]:>8.4f}')
print(f'{"CheXpert":<22} {best_chex["threshold"]:>9.2f} {best_chex["sensitivity"]:>11.4f} {best_chex["fnr"]:>8.4f} {best_chex["specificity"]:>11.4f} {best_chex["f1"]:>8.4f}')

Best NIH threshold     : 0.7
  Sensitivity          : 0.9220
  FNR                  : 0.0780
  Specificity          : 0.3134
  F1                   : 0.7386

Best CheXpert threshold: 0.7
  Sensitivity          : 0.8253
  FNR                  : 0.1747
  Specificity          : 0.3710
  F1                   : 0.6726

══ UPDATED FULL VALIDATION SUMMARY ═════════════════════════
Dataset                Threshold Sensitivity      FNR Specificity       F1
───────────────────────────────────────────────────────────────────────────
Kaggle                      0.70      0.9872   0.0128      0.7051   0.9123
NIH ChestX-ray14            0.70      0.9220   0.0780      0.3134   0.7386
CheXpert                    0.70      0.8253   0.1747      0.3710   0.6726


## restoring previous works without reloading-- except 1 2 cells


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import os

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        split_dir = os.path.join(root_dir, split)
        class_names = sorted(os.listdir(split_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
        for cls in class_names:
            cls_dir = os.path.join(split_dir, cls)
            for img_file in os.listdir(cls_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.images.append(os.path.join(cls_dir, img_file))
                    self.labels.append(self.class_to_idx[cls])
        print(f'[{split}] Loaded {len(self.images)} images | Classes: {self.class_to_idx}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.images[idx]).convert('RGB'))
        label = self.labels[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

def compute_class_weights(dataset):
    labels = np.array(dataset.labels)
    class_counts = np.bincount(labels)
    total = len(labels)
    weights = total / (len(class_counts) * class_counts)
    weights = torch.FloatTensor(weights).to(DEVICE)
    print(f'Class counts: {class_counts}')
    print(f'Class weights: {weights}')
    return weights

train_dataset = PneumoniaDataset(KAGGLE_PATH, 'train', transform=train_transforms)
val_dataset   = PneumoniaDataset(KAGGLE_PATH, 'val',   transform=val_test_transforms)
test_dataset  = PneumoniaDataset(KAGGLE_PATH, 'test',  transform=val_test_transforms)

CLASS_WEIGHTS = compute_class_weights(train_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

[train] Loaded 5216 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
[val] Loaded 16 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
[test] Loaded 624 images | Classes: {'NORMAL': 0, 'PNEUMONIA': 1}
Class counts: [1341 3875]
Class weights: tensor([1.9448, 0.6730], device='cuda:0')

Train batches : 163
Val batches   : 1
Test batches  : 20


In [8]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

class HybridPneumoniaLoss(nn.Module):
    def __init__(self, alpha, gamma, lambda_fn, class_weights):
        super(HybridPneumoniaLoss, self).__init__()
        self.alpha         = alpha
        self.gamma         = gamma
        self.lambda_fn     = lambda_fn
        self.class_weights = class_weights

    def forward(self, logits, targets):
        wbce = F.binary_cross_entropy_with_logits(
            logits, targets.float(),
            pos_weight=self.class_weights[1].unsqueeze(0)
        )
        probs      = torch.sigmoid(logits)
        focal_term = (1 - probs) ** self.gamma
        focal_loss = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction='none'
        )
        focal_loss = (focal_term * focal_loss).mean()
        preds      = (probs >= 0.5).float()
        fn_mask    = ((preds == 0) & (targets == 1)).float()
        fn_penalty = (fn_mask * (1 - probs)).mean()
        total_loss = self.alpha * wbce + focal_loss + self.lambda_fn * fn_penalty
        return total_loss, wbce.item(), focal_loss.item(), fn_penalty.item()

def build_model(architecture):
    if architecture == 'resnet50':
        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, 1)
    elif architecture == 'densenet121':
        model = models.densenet121(weights='IMAGENET1K_V1')
        model.classifier = nn.Linear(model.classifier.in_features, 1)
    return model.to(DEVICE)

BEST_ALPHA  = 1.0
BEST_GAMMA  = 0.5
BEST_LAMBDA = 5.0

print('Loss function and model builder ready ✅')

Loss function and model builder ready ✅


In [9]:
MODEL_PATH = '/kaggle/input/datasets/emmanuellakenneth/resnet50-hybrid-best-model/resnet50_hybrid_best.pth'

resnet_trained = build_model('resnet50')
resnet_trained.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
resnet_trained.eval()
print('Model loaded successfully ✅')
print(f'Model is on: {next(resnet_trained.parameters()).device}')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 167MB/s] 


Model loaded successfully ✅
Model is on: cuda:0


In [10]:
# Load NIH labels
nih_df = pd.read_csv(f'{NIH_PATH}/Data_Entry_2017.csv')

# Build image index
nih_image_folders = [f for f in os.listdir(NIH_PATH) if f.startswith('images_')]
nih_image_dict = {}
for folder in nih_image_folders:
    folder_path = os.path.join(NIH_PATH, folder, 'images')
    if os.path.exists(folder_path):
        for img_file in os.listdir(folder_path):
            nih_image_dict[img_file] = os.path.join(folder_path, img_file)

print(f'NIH images indexed: {len(nih_image_dict)}')

# Map labels
def map_nih_label(label):
    if label == 'No Finding':
        return 0
    elif 'Pneumonia' in label or 'Consolidation' in label:
        return 1
    else:
        return -1

nih_df['binary_label'] = nih_df['Finding Labels'].apply(map_nih_label)
nih_filtered = nih_df[nih_df['binary_label'] != -1].copy()
nih_filtered = nih_filtered[nih_filtered['Image Index'].isin(nih_image_dict.keys())].copy()
nih_filtered['image_path'] = nih_filtered['Image Index'].map(nih_image_dict)

# Sample 500 per class for fine-tuning
nih_normal_ft    = nih_filtered[nih_filtered['binary_label'] == 0].sample(500, random_state=42)
nih_pneumonia_ft = nih_filtered[nih_filtered['binary_label'] == 1].sample(500, random_state=42)
nih_ft_df        = pd.concat([nih_normal_ft, nih_pneumonia_ft]).reset_index(drop=True)

# Remaining for validation
nih_normal_val    = nih_filtered[nih_filtered['binary_label'] == 0].drop(nih_normal_ft.index).sample(3000, random_state=42)
nih_pneumonia_val = nih_filtered[nih_filtered['binary_label'] == 1].drop(nih_pneumonia_ft.index)
nih_val_df        = pd.concat([nih_normal_val, nih_pneumonia_val]).reset_index(drop=True)

print(f'NIH fine-tune set : {len(nih_ft_df)} images (500 normal, 500 pneumonia)')
print(f'NIH val set       : {len(nih_val_df)} images')

NIH images indexed: 112120
NIH fine-tune set : 1000 images (500 normal, 500 pneumonia)
NIH val set       : 8475 images


In [12]:
chex_train_df = pd.read_csv(f'{CHEXPERT_PATH}/train.csv')
chex_val_df   = pd.read_csv(f'{CHEXPERT_PATH}/valid.csv')

def map_chexpert_label(row):
    if row.get('No Finding') == 1.0:
        return 0
    elif row.get('Pneumonia') == 1.0:
        return 1
    else:
        return -1

chex_train_df['binary_label'] = chex_train_df.apply(map_chexpert_label, axis=1)
chex_val_df['binary_label']   = chex_val_df.apply(map_chexpert_label, axis=1)

chex_combined = pd.concat([chex_train_df, chex_val_df])
chex_filtered = chex_combined[chex_combined['binary_label'] != -1].copy()
chex_filtered['full_path'] = chex_filtered['Path'].apply(
    lambda x: os.path.join('/kaggle/input/datasets/ashery/chexpert',
                           x.replace('CheXpert-v1.0-small/', ''))
)
chex_filtered['exists'] = chex_filtered['full_path'].apply(os.path.exists)
chex_filtered = chex_filtered[chex_filtered['exists']].copy()

chex_normal    = chex_filtered[chex_filtered['binary_label'] == 0]
chex_pneumonia = chex_filtered[chex_filtered['binary_label'] == 1]

# Sample 500 per class for fine-tuning
chex_normal_ft    = chex_normal.sample(500, random_state=42)
chex_pneumonia_ft = chex_pneumonia.sample(500, random_state=42)
chex_ft_df        = pd.concat([chex_normal_ft, chex_pneumonia_ft]).reset_index(drop=True)

# Remaining for validation
chex_normal_val    = chex_normal.drop(chex_normal_ft.index).sample(3000, random_state=42)
chex_pneumonia_val = chex_pneumonia.drop(chex_pneumonia_ft.index)
chex_val_df2       = pd.concat([chex_normal_val, chex_pneumonia_val]).reset_index(drop=True)

print(f'CheXpert fine-tune set : {len(chex_ft_df)} images (500 normal, 500 pneumonia)')
print(f'CheXpert val set       : {len(chex_val_df2)} images')

CheXpert fine-tune set : 1000 images (500 normal, 500 pneumonia)
CheXpert val set       : 8547 images


In [13]:
class GenericDataset(Dataset):
    def __init__(self, dataframe, path_col, label_col, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.path_col  = path_col
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row      = self.dataframe.iloc[idx]
        img_path = row[self.path_col]
        label    = row[self.label_col]
        try:
            img = np.array(Image.open(img_path).convert('RGB'))
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            if self.transform:
                img = self.transform(image=img)['image']
            return img, label

# Combine NIH and CheXpert fine-tune sets
nih_ft_df['full_path']  = nih_ft_df['image_path']
chex_ft_df['full_path'] = chex_ft_df['full_path']

combined_ft_df = pd.concat([
    nih_ft_df[['full_path', 'binary_label']],
    chex_ft_df[['full_path', 'binary_label']]
]).reset_index(drop=True)

ft_dataset = GenericDataset(combined_ft_df, 'full_path', 'binary_label',
                             transform=train_transforms)
ft_loader  = DataLoader(ft_dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=2, pin_memory=True)

print(f'Combined fine-tune dataset: {len(combined_ft_df)} images')
print(f'Fine-tune batches         : {len(ft_loader)}')

Combined fine-tune dataset: 2000 images
Fine-tune batches         : 63


In [14]:
import copy

# Freeze all layers except the last block and FC layer
resnet_finetuned = copy.deepcopy(resnet_trained)

for name, param in resnet_finetuned.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in resnet_finetuned.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet_finetuned.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

loss_fn   = HybridPneumoniaLoss(BEST_ALPHA, BEST_GAMMA, BEST_LAMBDA, CLASS_WEIGHTS)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_finetuned.parameters()),
    lr=1e-5  # lower LR for fine-tuning
)

FT_EPOCHS = 5
print(f'\nFine-tuning for {FT_EPOCHS} epochs on combined NIH + CheXpert sample...\n')

resnet_finetuned.train()
for epoch in range(FT_EPOCHS):
    epoch_loss = 0.0
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = resnet_finetuned(imgs).squeeze(1)
        loss, _, _, _ = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'Epoch [{epoch+1}/{FT_EPOCHS}] Loss: {epoch_loss/len(ft_loader):.4f}')

torch.save(resnet_finetuned.state_dict(),
           f'{CHECKPOINT_DIR}/resnet50_finetuned.pth')
print(f'\nFine-tuned model saved ✅')

Trainable params: 14,966,785 / 23,510,081

Fine-tuning for 5 epochs on combined NIH + CheXpert sample...

Epoch [1/5] Loss: 1.6912
Epoch [2/5] Loss: 1.2053
Epoch [3/5] Loss: 1.0752
Epoch [4/5] Loss: 1.0247
Epoch [5/5] Loss: 1.0077

Fine-tuned model saved ✅


In [16]:
BEST_THRESHOLD = 0.70

In [18]:
def evaluate_dataset(model, loader, dataset_name):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            logits = model(imgs).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().tolist()
            preds  = [1 if p >= BEST_THRESHOLD else 0 for p in probs]
            all_probs  += probs
            all_preds  += preds
            all_labels += labels.tolist()

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0,1]).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    fnr         = fn / (fn + tp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    f1          = f1_score(all_labels, all_preds)
    accuracy    = accuracy_score(all_labels, all_preds)
    roc_auc     = roc_auc_score(all_labels, all_probs)
    precision, recall, _ = precision_recall_curve(all_labels, all_probs)
    auprc       = auc(recall, precision)

    print(f'\n── {dataset_name} ──────────────────────────────')
    print(f'  Sensitivity : {sensitivity:.4f}')
    print(f'  FNR         : {fnr:.4f}')
    print(f'  Specificity : {specificity:.4f}')
    print(f'  F1          : {f1:.4f}')
    print(f'  Accuracy    : {accuracy:.4f}')
    print(f'  ROC-AUC     : {roc_auc:.4f}')
    print(f'  AUPRC       : {auprc:.4f}')
    print(f'  TP={tp} | TN={tn} | FP={fp} | FN={fn}')
    return sensitivity, fnr, specificity, f1, auprc

# Build NIH and CheXpert val loaders
nih_val_dataset  = GenericDataset(nih_val_df,  'image_path', 'binary_label',
                                   transform=val_test_transforms)
chex_val_dataset = GenericDataset(chex_val_df2, 'full_path', 'binary_label',
                                   transform=val_test_transforms)

nih_val_loader  = DataLoader(nih_val_dataset,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
chex_val_loader = DataLoader(chex_val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

print('Evaluating fine-tuned model on all datasets...')
s1, f1_1, sp1, f1s1, au1 = evaluate_dataset(resnet_finetuned, test_loader,     'Kaggle Test Set')
s2, f1_2, sp2, f1s2, au2 = evaluate_dataset(resnet_finetuned, nih_val_loader,  'NIH ChestX-ray14')
s3, f1_3, sp3, f1s3, au3 = evaluate_dataset(resnet_finetuned, chex_val_loader, 'CheXpert')

print(f'\n══ FINAL COMPARISON: Before vs After Fine-tuning ════════════')
print(f'{"Dataset":<22} {"Before Sens":>11} {"After Sens":>11} {"Before Spec":>12} {"After Spec":>11}')
print('─' * 72)
print(f'{"Kaggle":<22} {0.9872:>11.4f} {s1:>11.4f} {0.7051:>12.4f} {sp1:>11.4f}')
print(f'{"NIH ChestX-ray14":<22} {0.9220:>11.4f} {s2:>11.4f} {0.3134:>12.4f} {sp2:>11.4f}')
print(f'{"CheXpert":<22} {0.8253:>11.4f} {s3:>11.4f} {0.3710:>12.4f} {sp3:>11.4f}')

Evaluating fine-tuned model on all datasets...


NameError: name 'resnet_finetuned2' is not defined

In [19]:
import copy

resnet_finetuned2 = copy.deepcopy(resnet_trained)

# Freeze all layers except layer4 and fc
for name, param in resnet_finetuned2.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# Higher lambda to protect sensitivity, lower LR
loss_fn_ft = HybridPneumoniaLoss(1.0, 0.5, 10.0, CLASS_WEIGHTS)
optimizer  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_finetuned2.parameters()),
    lr=1e-6  # much lower LR
)

FT_EPOCHS = 3  # fewer epochs
print(f'Fine-tuning with λ=10.0 and LR=1e-6 for {FT_EPOCHS} epochs...\n')

resnet_finetuned2.train()
for epoch in range(FT_EPOCHS):
    epoch_loss = 0.0
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = resnet_finetuned2(imgs).squeeze(1)
        loss, _, _, _ = loss_fn_ft(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'Epoch [{epoch+1}/{FT_EPOCHS}] Loss: {epoch_loss/len(ft_loader):.4f}')

torch.save(resnet_finetuned2.state_dict(),
           f'{CHECKPOINT_DIR}/resnet50_finetuned2.pth')
print(f'\nFine-tuned model v2 saved ✅')

Fine-tuning with λ=10.0 and LR=1e-6 for 3 epochs...

Epoch [1/3] Loss: 2.2113
Epoch [2/3] Loss: 2.2042
Epoch [3/3] Loss: 2.0268

Fine-tuned model v2 saved ✅


In [20]:
def evaluate_dataset(model, loader, dataset_name):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            logits = model(imgs).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().tolist()
            preds  = [1 if p >= BEST_THRESHOLD else 0 for p in probs]
            all_probs  += probs
            all_preds  += preds
            all_labels += labels.tolist()

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0,1]).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    fnr         = fn / (fn + tp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    f1          = f1_score(all_labels, all_preds)
    accuracy    = accuracy_score(all_labels, all_preds)
    roc_auc     = roc_auc_score(all_labels, all_probs)
    precision, recall, _ = precision_recall_curve(all_labels, all_probs)
    auprc       = auc(recall, precision)

    print(f'\n── {dataset_name} ──────────────────────────────')
    print(f'  Sensitivity : {sensitivity:.4f}')
    print(f'  FNR         : {fnr:.4f}')
    print(f'  Specificity : {specificity:.4f}')
    print(f'  F1          : {f1:.4f}')
    print(f'  Accuracy    : {accuracy:.4f}')
    print(f'  ROC-AUC     : {roc_auc:.4f}')
    print(f'  AUPRC       : {auprc:.4f}')
    print(f'  TP={tp} | TN={tn} | FP={fp} | FN={fn}')
    return sensitivity, fnr, specificity, f1, auprc

# Build NIH and CheXpert val loaders
nih_val_dataset  = GenericDataset(nih_val_df,  'image_path', 'binary_label',
                                   transform=val_test_transforms)
chex_val_dataset = GenericDataset(chex_val_df2, 'full_path', 'binary_label',
                                   transform=val_test_transforms)

nih_val_loader  = DataLoader(nih_val_dataset,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
chex_val_loader = DataLoader(chex_val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

print('Evaluating fine-tuned model on all datasets...')
s1, f1_1, sp1, f1s1, au1 = evaluate_dataset(resnet_finetuned2, test_loader,     'Kaggle Test Set')
s2, f1_2, sp2, f1s2, au2 = evaluate_dataset(resnet_finetuned2, nih_val_loader,  'NIH ChestX-ray14')
s3, f1_3, sp3, f1s3, au3 = evaluate_dataset(resnet_finetuned2, chex_val_loader, 'CheXpert')

print(f'\n══ FINAL COMPARISON: Before vs After Fine-tuning ════════════')
print(f'{"Dataset":<22} {"Before Sens":>11} {"After Sens":>11} {"Before Spec":>12} {"After Spec":>11}')
print('─' * 72)
print(f'{"Kaggle":<22} {0.9872:>11.4f} {s1:>11.4f} {0.7051:>12.4f} {sp1:>11.4f}')
print(f'{"NIH ChestX-ray14":<22} {0.9220:>11.4f} {s2:>11.4f} {0.3134:>12.4f} {sp2:>11.4f}')
print(f'{"CheXpert":<22} {0.8253:>11.4f} {s3:>11.4f} {0.3710:>12.4f} {sp3:>11.4f}')

Evaluating fine-tuned model on all datasets...

── Kaggle Test Set ──────────────────────────────
  Sensitivity : 0.9846
  FNR         : 0.0154
  Specificity : 0.6709
  F1          : 0.9025
  Accuracy    : 0.8670
  ROC-AUC     : 0.9597
  AUPRC       : 0.9725
  TP=384 | TN=157 | FP=77 | FN=6

── NIH ChestX-ray14 ──────────────────────────────
  Sensitivity : 0.9123
  FNR         : 0.0877
  Specificity : 0.3420
  F1          : 0.8028
  Accuracy    : 0.7104
  ROC-AUC     : 0.7390
  AUPRC       : 0.8139
  TP=4995 | TN=1026 | FP=1974 | FN=480

── CheXpert ──────────────────────────────
  Sensitivity : 0.8094
  FNR         : 0.1906
  Specificity : 0.4043
  F1          : 0.7595
  Accuracy    : 0.6673
  ROC-AUC     : 0.6920
  AUPRC       : 0.8044
  TP=4490 | TN=1213 | FP=1787 | FN=1057

══ FINAL COMPARISON: Before vs After Fine-tuning ════════════
Dataset                Before Sens  After Sens  Before Spec  After Spec
────────────────────────────────────────────────────────────────────────
Kag